In [2]:
import math
from typing import Any

from bloqade import squin, tsim
from bloqade.types import MeasurementResult, Qubit
from kirin.dialects.ilist import IList

Register = IList[Qubit, Any]
Measurement = IList[MeasurementResult, Any]

def show_circuit(squin_kernel):
    @squin.kernel
    def _to_visualize():
        _ = squin_kernel()

    return tsim.Circuit(_to_visualize).diagram(height=400)

## HH Gate

Applying Hadamard twice returns the qubit to its original state, since $H^2 = I$.

In [3]:
@squin.kernel
def double_hadamard() -> Measurement:
    qubit = squin.qalloc(1)
    squin.h(qubit[0])
    squin.h(qubit[0])
    bits = squin.broadcast.measure(qubit)
    return bits

In [4]:
show_circuit(double_hadamard)

## HSH Gate

**Why the output is a Pauli eigenstate.**

The $S$ gate is *Clifford*: it maps every Pauli operator to another Pauli operator under conjugation ($SXS^\dagger = Y$, $SZS^\dagger = Z$). Because the full Clifford group is closed under composition, $HSH$ is also Clifford. A key property of Clifford circuits is that they map *stabilizer states* (Pauli eigenstates) to stabilizer states. Starting from $|0\rangle$, which is a $+1$ eigenstate of $Z$, we stay in a stabilizer state at every step:

$$
|0\rangle \xrightarrow{H} |{+}\rangle \xrightarrow{S} |{+i}\rangle \xrightarrow{H} e^{i\pi/4}|{-i}\rangle
$$

$|{-i}\rangle$ is the $-1$ eigenstate of the Pauli $Y$ operator — still a Pauli eigenstate (up to a global phase that has no physical significance).

In [5]:
@squin.kernel
def hsh() -> Measurement:
    qubit = squin.qalloc(1)
    squin.h(qubit[0])
    squin.s(qubit[0])
    squin.h(qubit[0])
    bits = squin.broadcast.measure(qubit)
    return bits

In [6]:
show_circuit(hsh)

## HTH Gate

**Why the output is NOT a Pauli eigenstate.**

The $T$ gate is *non-Clifford*: $T = \operatorname{diag}(1,\, e^{i\pi/4})$, so it rotates $X$ into $(X + Y)/\sqrt{2}$ — a direction that is not in the Pauli group. Because $T$ is outside the Clifford group, $HTH$ can escape the set of stabilizer states entirely. Starting from $|0\rangle$:

$$
|0\rangle \xrightarrow{H} |{+}\rangle \xrightarrow{T} \frac{|0\rangle + e^{i\pi/4}|1\rangle}{\sqrt{2}} \xrightarrow{H}
\frac{(1+e^{i\pi/4})|0\rangle + (1-e^{i\pi/4})|1\rangle}{2}
$$

The resulting state $\frac{(1+e^{i\pi/4})|0\rangle + (1-e^{i\pi/4})|1\rangle}{2}$ is not an eigenstate of $X$, $Y$, or $Z$ — it is a *magic state*, the resource that makes $T$ gates classically hard to simulate and the key ingredient in universal quantum computation beyond the Clifford group.

In [7]:
@squin.kernel
def hth() -> Measurement:
    qubit = squin.qalloc(1)
    squin.h(qubit[0])
    squin.t(qubit[0])
    squin.h(qubit[0])
    bits = squin.broadcast.measure(qubit)
    return bits

In [8]:
show_circuit(hth)

## Bell State

The standard Bell state $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$ is prepared by applying a Hadamard to qubit 0 to create a superposition, then entangling both qubits with a CNOT.

In [9]:
@squin.kernel
def bell_state() -> Measurement:
    qubits = squin.qalloc(2)
    squin.h(qubits[0])
    squin.cx(qubits[0], qubits[1])
    bits = squin.broadcast.measure(qubits)
    return bits

In [10]:
show_circuit(bell_state)

## Bell State with T Gate Before CNOT

Inserting a $T$ gate on qubit 0 between the Hadamard and the CNOT rotates the control qubit into a magic state before entangling. The resulting two-qubit state is no longer a stabilizer state — it lies outside the reach of any Clifford circuit and cannot be efficiently simulated by the stabilizer formalism.

In [11]:
@squin.kernel
def bell_state_t() -> Measurement:
    qubits = squin.qalloc(2)
    squin.h(qubits[0])
    squin.t(qubits[0])
    squin.cx(qubits[0], qubits[1])
    bits = squin.broadcast.measure(qubits)
    return bits

In [12]:
show_circuit(bell_state_t)

## Bell State with S Gate Before CNOT

Inserting an $S$ gate on qubit 0 between the Hadamard and the CNOT is still a Clifford operation. The control qubit is rotated to $|{+i}\rangle$ before the entangling gate, producing a two-qubit stabilizer state that is a different Bell-basis state but remains within the stabilizer formalism.

In [13]:
@squin.kernel
def bell_state_s() -> Measurement:
    qubits = squin.qalloc(2)
    squin.h(qubits[0])
    squin.s(qubits[0])
    squin.cx(qubits[0], qubits[1])
    bits = squin.broadcast.measure(qubits)
    return bits

In [14]:
show_circuit(bell_state_s)

## $R_z(\pi/2)$ = S (Clifford)

$R_z(\pi/2) = e^{-i\pi/4}\,S$, so up to a global phase $R_z(\pi/2)$ is exactly the $S$ gate. Because $S$ is Clifford, no $T$ gates are needed — this rotation sits inside the Clifford group.

In [15]:
@squin.kernel
def rz_half_pi() -> Measurement:
    qubit = squin.qalloc(1)
    squin.rz(math.pi / 2, qubit[0])
    bits = squin.broadcast.measure(qubit)
    return bits

In [16]:
show_circuit(rz_half_pi)

In [17]:
# Equivalent Clifford circuit: S
@squin.kernel
def rz_half_pi_clifford() -> Measurement:
    qubit = squin.qalloc(1)
    squin.s(qubit[0])
    bits = squin.broadcast.measure(qubit)
    return bits

In [18]:
show_circuit(rz_half_pi_clifford)

## $R_z(\pi/4)$ = T (non-Clifford)

$R_z(\pi/4) = e^{-i\pi/8}\,T$, so up to a global phase $R_z(\pi/4)$ is exactly the $T$ gate. Unlike $S$ and $Z$, the $T$ gate is **non-Clifford**: it cannot be expressed using only Clifford gates. It is the fundamental non-Clifford resource — any angle that is not a multiple of $\pi/2$ requires at least one $T$ gate to implement exactly.

In [19]:
@squin.kernel
def rz_quarter_pi() -> Measurement:
    qubit = squin.qalloc(1)
    squin.rz(math.pi / 4, qubit[0])
    bits = squin.broadcast.measure(qubit)
    return bits

In [20]:
show_circuit(rz_quarter_pi)

In [21]:
# Equivalent non-Clifford circuit: T
@squin.kernel
def rz_quarter_pi_t() -> Measurement:
    qubit = squin.qalloc(1)
    squin.t(qubit[0])
    bits = squin.broadcast.measure(qubit)
    return bits

In [22]:
show_circuit(rz_quarter_pi_t)

## $R_z(\pi/8)$ — No exact Clifford+T decomposition

$R_z(\pi/8) = e^{-i\pi/16}\,\begin{pmatrix}1&0\\0&e^{i\pi/8}\end{pmatrix}$. Unlike $R_z(\pi/4) = T$, $R_z(\pi/2) = S$, and $R_z(\pi) = Z$, this angle **cannot be exactly decomposed** into a finite sequence of Clifford + $T$ gates. The Clifford+$T$ gateset can only represent $R_z$ exactly for angles that are integer multiples of $\pi/4$. For all other angles — including $\pi/8$, $\pi/16$, $\pi/32$ — you need an *approximation* algorithm such as Solovay–Kitaev or Ross–Selinger, which produces a Clifford+$T$ sequence that gets arbitrarily close but uses more $T$ gates the higher the precision required.

In [23]:
@squin.kernel
def rz_eighth_pi() -> Measurement:
    qubit = squin.qalloc(1)
    squin.rz(math.pi / 8, qubit[0])
    bits = squin.broadcast.measure(qubit)
    return bits

In [24]:
show_circuit(rz_eighth_pi)

## $R_z(\pi/16)$ — No exact Clifford+T decomposition

$R_z(\pi/16)$ requires an even finer rotation than $\pi/8$ and is equally outside the exact Clifford+$T$ representable set. Each halving of the angle doubles the $T$-count needed for the same approximation precision, making very small angles increasingly expensive to implement fault-tolerantly.

In [25]:
@squin.kernel
def rz_sixteenth_pi() -> Measurement:
    qubit = squin.qalloc(1)
    squin.rz(math.pi / 16, qubit[0])
    bits = squin.broadcast.measure(qubit)
    return bits

In [26]:
show_circuit(rz_sixteenth_pi)

## $R_z(\pi/32)$ — No exact Clifford+T decomposition

$R_z(\pi/32)$ continues the same pattern. In fault-tolerant quantum computing, the $T$ gate is the most expensive primitive because it requires *magic state distillation*. The fact that $\pi/8$, $\pi/16$, $\pi/32$ all fall outside the exact Clifford+$T$ set is why minimizing $T$-count is a central concern in quantum circuit compilation — every non-Clifford rotation incurs a cost that grows with precision.

In [27]:
@squin.kernel
def rz_thirtysecond_pi() -> Measurement:
    qubit = squin.qalloc(1)
    squin.rz(math.pi / 32, qubit[0])
    bits = squin.broadcast.measure(qubit)
    return bits

In [28]:
show_circuit(rz_thirtysecond_pi)